# Prior-Study Viability

Goal: decide whether incorporating a patient's previous CXR study into the CaMCheX model is worth the engineering effort.

We check:
1. **Coverage** — what fraction of studies have a recorded prior?
2. **History depth** — how many priors per patient (longitudinal richness)?
3. **Time gap** — days between current and prior study (is the prior still informative?)
4. **Label dynamics** — do labels actually *change* between visits? (If priors == current, the model just memorizes the prior.)
5. **Per-class priors-helpfulness proxy** — conditional label rates given the prior had the same finding.
6. **Split safety** — are prior studies always in the same split as the current (no leakage)?

In [ ]:
import os
os.environ['CURRENT_NOTEBOOK_NAME'] = 'prior-study-viability'
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from utils import *


In [ ]:
root_dir, data_dir = get_notebook_paths()
DATA_CAMCHEX = data_dir / "camchex"
MIMIC_CXR_JPG = data_dir / "MIMIC-CXR-JPG"

print(Path.cwd())
print(root_dir)
print(data_dir)


In [ ]:
CLASSES = [
    'Atelectasis','Calcification of the Aorta','Cardiomegaly','Consolidation','Edema',
    'Emphysema','Enlarged Cardiomediastinum','Fibrosis','Fracture','Hernia',
    'Infiltration','Lung Lesion','Lung Opacity','Mass','No Finding','Nodule',
    'Pleural Effusion','Pleural Other','Pleural Thickening','Pneumomediastinum',
    'Pneumonia','Pneumoperitoneum','Pneumothorax','Subcutaneous Emphysema',
    'Support Devices','Tortuous Aorta'
]

frames = []
for split in ["train", "development", "test"]:
    p = DATA_CAMCHEX / f"02_{split}.csv"
    df = pd.read_csv(p, low_memory=False)
    df["split"] = split
    frames.append(df)
df_row = pd.concat(frames, ignore_index=True)

# Collapse to study-level (one row per study)
df = df_row.drop_duplicates("study_id").copy()
df["StudyDateTime"] = pd.to_datetime(df["StudyDateTime"], errors="coerce")
df["has_prior"] = df["PreviousStudy"].notna()
print("row-level:", len(df_row), "study-level:", len(df))
df.head(3)


## 1. Coverage — fraction of studies with a prior

In [ ]:
cov = df.groupby("split")["has_prior"].agg(["sum", "count"])
cov["ratio"] = cov["sum"] / cov["count"]
cov.loc["ALL"] = [df["has_prior"].sum(), len(df), df["has_prior"].mean()]
print(cov)

fig, ax = plt.subplots(figsize=(6, 3.5))
cov_plot = cov.drop("ALL").reset_index()
sns.barplot(data=cov_plot, x="split", y="ratio", ax=ax)
for i, r in enumerate(cov_plot["ratio"]):
    ax.text(i, r + 0.01, f"{r:.1%}", ha="center")
ax.set_ylim(0, 1)
ax.set_ylabel("Fraction of studies with prior")
ax.set_title("Prior-study coverage by split")
plt.tight_layout(); plt.show()


## 2. History depth — studies per patient

In [ ]:
studies_per_patient = df.groupby("subject_id")["study_id"].nunique()
print(studies_per_patient.describe())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
spp_clip = studies_per_patient.clip(upper=15)
sns.histplot(spp_clip, bins=range(1, 17), discrete=True, ax=axes[0])
axes[0].set_xlabel("# studies per patient (clipped at 15)")
axes[0].set_ylabel("# patients")
axes[0].set_title("Distribution of studies per patient")

buckets = pd.cut(studies_per_patient, bins=[0, 1, 2, 3, 5, 10, 1e9],
                 labels=["1", "2", "3", "4-5", "6-10", "11+"])
buckets.value_counts(sort=False).plot(kind="bar", ax=axes[1])
axes[1].set_ylabel("# patients")
axes[1].set_title("Bucketed history depth")
for p in axes[1].patches:
    axes[1].annotate(int(p.get_height()), (p.get_x() + p.get_width()/2, p.get_height()),
                     ha="center", va="bottom")
plt.tight_layout(); plt.show()


## 3. Time gap to prior
If the prior was years ago, it's probably stale; if it was minutes ago (same admission), it's nearly identical.

In [ ]:
date_lookup = df.set_index("study_id")["StudyDateTime"].to_dict()

df["prev_date"] = df["PreviousStudy"].map(lambda prev_id: lookup_previous_study_date(prev_id, date_lookup))
df["days_since_prev"] = (df["StudyDateTime"] - df["prev_date"]).dt.total_seconds() / 86400

gaps = df["days_since_prev"].dropna()
print("gaps with both dates known:", len(gaps), "of", df["has_prior"].sum(), "priors")
print(gaps.describe(percentiles=[.1, .25, .5, .75, .9, .99]))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(gaps.clip(0, 365*3), bins=60, ax=axes[0])
axes[0].set_xlabel("Days since prior (clipped at 3 years)")
axes[0].set_title("Time-gap distribution (linear)")

sns.histplot(np.log1p(gaps[gaps >= 0]), bins=60, ax=axes[1])
axes[1].set_xlabel("log(1 + days since prior)")
axes[1].set_title("Time-gap distribution (log)")
plt.tight_layout(); plt.show()

# Bucket gaps into clinically meaningful windows
buckets = pd.cut(gaps, bins=[-0.5, 1, 7, 30, 180, 365, 365*3, np.inf],
                 labels=["≤1d", "2-7d", "8-30d", "1-6mo", "6-12mo", "1-3y", ">3y"])
fig, ax = plt.subplots(figsize=(7, 3.5))
buckets.value_counts(sort=False).plot(kind="bar", ax=ax)
ax.set_title("Time gap buckets")
ax.set_ylabel("# studies")
for p in ax.patches:
    ax.annotate(int(p.get_height()), (p.get_x() + p.get_width()/2, p.get_height()),
                ha="center", va="bottom")
plt.tight_layout(); plt.show()


## 4. Label dynamics — do findings change between visits?
If labels never change between current and prior, the prior is just a leak / shortcut. We want some change, but not total chaos.

In [ ]:
label_lookup = df.set_index("study_id")[CLASSES].astype(float)

paired = df.dropna(subset=["PreviousStudy"]).copy()
paired["PreviousStudy_int"] = paired["PreviousStudy"].astype("Int64")
paired = paired[paired["PreviousStudy_int"].isin(label_lookup.index)]

cur = label_lookup.loc[paired["study_id"].values].fillna(0).values
prv = label_lookup.loc[paired["PreviousStudy_int"].astype(int).values].fillna(0).values
cur_b = (cur > 0).astype(int)
prv_b = (prv > 0).astype(int)

agree = (cur_b == prv_b).mean(axis=0)
new_pos = ((cur_b == 1) & (prv_b == 0)).mean(axis=0)
lost_pos = ((cur_b == 0) & (prv_b == 1)).mean(axis=0)
prev_pos = prv_b.mean(axis=0)
cur_pos = cur_b.mean(axis=0)

dyn = pd.DataFrame({
    "class": CLASSES,
    "prev_prevalence": prev_pos,
    "cur_prevalence": cur_pos,
    "agreement": agree,
    "new_positive": new_pos,
    "lost_positive": lost_pos,
}).sort_values("cur_prevalence", ascending=False)
dyn


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
order = dyn.sort_values("agreement").reset_index(drop=True)
axes[0].barh(order["class"], order["agreement"], color="steelblue")
axes[0].axvline(0.9, color="red", ls="--", lw=1, label="0.90 (very static)")
axes[0].axvline(0.5, color="orange", ls="--", lw=1, label="0.50 (coin flip)")
axes[0].set_xlabel("Agreement (current label == prior label)")
axes[0].set_title("How often does the label stay the same?")
axes[0].legend()

flip = order[["class", "new_positive", "lost_positive"]].set_index("class")
flip.plot(kind="barh", stacked=True, ax=axes[1], color=["#2ca02c", "#d62728"])
axes[1].set_xlabel("Fraction of pairs")
axes[1].set_title("Label transitions (new positive vs. lost positive)")
plt.tight_layout(); plt.show()


## 5. Is the prior actually informative?
P(current label = 1 | prior label = 1) vs. P(current label = 1 | prior label = 0).
A large gap means the prior carries real signal for that class.

In [ ]:
eps = 1e-9
p_cur_given_prv1 = (cur_b * prv_b).sum(0) / (prv_b.sum(0) + eps)
p_cur_given_prv0 = (cur_b * (1 - prv_b)).sum(0) / ((1 - prv_b).sum(0) + eps)
lift = p_cur_given_prv1 - p_cur_given_prv0

info = pd.DataFrame({
    "class": CLASSES,
    "P(cur=1 | prv=1)": p_cur_given_prv1,
    "P(cur=1 | prv=0)": p_cur_given_prv0,
    "lift": lift,
    "base_rate": cur_pos,
}).sort_values("lift", ascending=False)

fig, ax = plt.subplots(figsize=(8, 7))
y = np.arange(len(info))
ax.barh(y, info["P(cur=1 | prv=1)"], color="#2ca02c", label="prv=1")
ax.barh(y, info["P(cur=1 | prv=0)"], color="#d62728", alpha=0.6, label="prv=0")
ax.set_yticks(y); ax.set_yticklabels(info["class"])
ax.invert_yaxis()
ax.set_xlabel("P(current = 1)")
ax.set_title("Prior label informativeness per class")
ax.legend()
plt.tight_layout(); plt.show()
info


## 6. Split safety — is the prior always in the same split?
If a `test` sample's prior lives in `train`, we'd leak training images into eval. The dataset should keep all of a patient's studies in one split.

In [ ]:
split_lookup = df.set_index("study_id")["split"].to_dict()
paired = df.dropna(subset=["PreviousStudy"]).copy()
paired["prev_split"] = paired["PreviousStudy"].map(
    lambda x: split_lookup.get(int(x)) if pd.notna(x) else None
)
paired = paired.dropna(subset=["prev_split"])
leak = pd.crosstab(paired["split"], paired["prev_split"])
print(leak)
leak_norm = leak.div(leak.sum(axis=1), axis=0)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(leak_norm, annot=True, fmt=".2%", cmap="Blues", ax=ax)
ax.set_title("P(prior split | current split)")
plt.tight_layout(); plt.show()


## 7. View availability — does the prior have a frontal view we can encode?
If most priors are single-view AP/PA only, that's fine — means cheap to encode.

In [ ]:
views_per_study = df_row.groupby("study_id")["ViewPosition"].apply(
    lambda s: set(str(v).upper() for v in s.dropna())
)
n_views = views_per_study.apply(len)
print(n_views.describe())

fig, ax = plt.subplots(figsize=(6, 3.5))
sns.histplot(n_views.clip(upper=6), bins=range(1, 8), discrete=True, ax=ax)
ax.set_xlabel("# distinct view positions per study (clipped at 6)")
ax.set_ylabel("# studies")
ax.set_title("Views per study")
plt.tight_layout(); plt.show()

from collections import Counter
view_counts = Counter(v for s in views_per_study for v in s)
print(pd.Series(view_counts).sort_values(ascending=False).head(10))


## Verdict checklist

Looking at the plots above, the prior-study direction is **viable** when:
- **Coverage** ≳ 50%. If most studies have no prior, the model has to learn two modes (with/without prior) on top of the harder task — still doable, but the ablation gain is muted.
- **History depth** has a heavy tail. Many patients with 3+ visits = clear case for extending beyond just-prior-one.
- **Time gaps** are not all >1y. Sub-month gaps are the cleanest case; if median is 1–6mo we're in good shape.
- **Label agreement** is high but not 1.0. ~0.7–0.95 per class is the sweet spot — priors are predictive but not identical, so the model can't just copy.
- **Lift** ( P(cur|prv=1) − P(cur|prv=0) ) is large for chronic findings (Cardiomegaly, Calcification of the Aorta, Tortuous Aorta, Support Devices). That's the prior's value proposition.
- **Split safety**: diagonal of the leak heatmap should be ~100%. If not, fix the split before training.
